# Bellabeat Marketing Analysis

**Google Data Analytics Capstone — Case Study 2**

*By Jake*

---

Bellabeat makes wellness products for women — an app, a few smart trackers, and a water bottle. They wanted to know how people are actually using smart fitness devices in general, and what that could mean for marketing the Bellabeat app.

I worked through two months of public FitBit data from 35 users to find out. This notebook walks through the cleaning, analysis, and chart-making behind the final report and deck.

**The headlines:**
- Users split pretty evenly across four activity tiers, so there's no real "average user" to market to.
- People spend about 79% of their tracked day sitting — even the active ones.
- Sitting time is the strongest thing connected to bad sleep (correlation of -0.64). Steps barely matter.
- A third of sleep-tracking users are getting under 6 hours a night, but their sleep itself is efficient. They're just not going to bed early enough.
- Activity peaks at 7 PM and dips between 1 and 4 PM — useful if Bellabeat wants to time notifications well.

> **A note on AI assistance:** This analysis was completed with help from Claude (Anthropic's AI assistant) for code structure, technical guidance, and writing polish. All analytical decisions — which product to focus on, how to handle data quality issues, what findings to surface — were mine. I ran every line of code, reviewed all outputs, and validated the findings against the data.

## 1. Setup

Standard imports plus the pathlib `Path` for handling the dataset's nested folder structure. The dataset comes in two folders covering different date ranges (March 12 – April 11, then April 12 – May 12, 2016), and each parent folder has another folder inside it where the actual CSVs live.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import Patch
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

# Update BASE_DIR to wherever your project lives
BASE_DIR  = Path(r"C:\Users\jakec\Documents\Coursera Case Studies\Case Study 2")
RAW_DIR   = BASE_DIR / "01_raw_data"
CLEAN_DIR = BASE_DIR / "02_cleaned_data"
VIZ_DIR   = BASE_DIR / "04_visualizations"

FOLDER_1 = RAW_DIR / "mturkfitbit_export_3.12.16-4.11.16" / "Fitabase Data 3.12.16-4.11.16"
FOLDER_2 = RAW_DIR / "mturkfitbit_export_4.12.16-5.12.16" / "Fitabase Data 4.12.16-5.12.16"

CLEAN_DIR.mkdir(exist_ok=True)
VIZ_DIR.mkdir(exist_ok=True)

print(f"Folder 1 exists: {FOLDER_1.exists()}")
print(f"Folder 2 exists: {FOLDER_2.exists()}")

## 2. What's in the data?

Quick recon before loading anything. The two folders have different files — Folder 2 has more (it added daily summaries, hourly breakdowns, and the sleep file). Worth checking what overlaps so we know what we can combine.

In [ ]:
folder_1_files = sorted(FOLDER_1.glob("*.csv"))
folder_2_files = sorted(FOLDER_2.glob("*.csv"))

print(f"Folder 1: {len(folder_1_files)} CSVs")
print(f"Folder 2: {len(folder_2_files)} CSVs")

files_1 = {f.name for f in folder_1_files}
files_2 = {f.name for f in folder_2_files}
print(f"\nIn both folders: {len(files_1 & files_2)} files")
print(f"Only in Folder 2: {sorted(files_2 - files_1)}")

**Important catch:** `sleepDay_merged.csv` is only in Folder 2. That means our sleep analysis is limited to 31 days (April 12 – May 12), not the full 61. Worth flagging upfront because it caps how confident we can be in any sleep findings.

## 3. Load the four files we'll actually use

Out of 18 files in Folder 2, we only need four for a marketing-focused analysis:

- `dailyActivity_merged.csv` — the workhorse (steps, distance, calories, intensity minutes)
- `sleepDay_merged.csv` — daily sleep totals
- `hourlySteps_merged.csv` — for the time-of-day pattern
- `weightLogInfo_merged.csv` — mostly to show how few people use it

Skipping the minute-level files. They're huge and don't add anything the daily/hourly files don't already cover.

In [ ]:
# Combine both folders for files that exist in both
daily_activity_1 = pd.read_csv(FOLDER_1 / "dailyActivity_merged.csv")
daily_activity_2 = pd.read_csv(FOLDER_2 / "dailyActivity_merged.csv")
daily_activity   = pd.concat([daily_activity_1, daily_activity_2], ignore_index=True)

# Sleep is Folder 2 only
sleep_day = pd.read_csv(FOLDER_2 / "sleepDay_merged.csv")

hourly_steps_1 = pd.read_csv(FOLDER_1 / "hourlySteps_merged.csv")
hourly_steps_2 = pd.read_csv(FOLDER_2 / "hourlySteps_merged.csv")
hourly_steps   = pd.concat([hourly_steps_1, hourly_steps_2], ignore_index=True)

weight_log_1 = pd.read_csv(FOLDER_1 / "weightLogInfo_merged.csv")
weight_log_2 = pd.read_csv(FOLDER_2 / "weightLogInfo_merged.csv")
weight_log   = pd.concat([weight_log_1, weight_log_2], ignore_index=True)

print("Loaded data:")
print(f"  Daily activity: {len(daily_activity):>6} rows, {daily_activity['Id'].nunique()} unique users")
print(f"  Sleep day:      {len(sleep_day):>6} rows, {sleep_day['Id'].nunique()} unique users")
print(f"  Hourly steps:   {len(hourly_steps):>6} rows, {hourly_steps['Id'].nunique()} unique users")
print(f"  Weight log:     {len(weight_log):>6} rows, {weight_log['Id'].nunique()} unique users")

Already two findings before any cleaning:

1. **35 users** show up in activity data (the case study brief said 30 — actual sample is slightly bigger).
2. **Only 24 of 35 users (69%)** logged sleep. **Only 13 of 35 users (37%)** ever logged weight, with just 100 entries total across two months. Most users either didn't engage with these features or gave up quickly. Already useful for the recommendations.

## 4. Investigate duplicates BEFORE dropping anything

This step matters. Blindly running `drop_duplicates()` can erase real data if you don't know what's actually duplicated.

In [ ]:
print("Duplicate rows by (Id, date):")
print(f"  Daily activity: {daily_activity.duplicated(subset=['Id','ActivityDate']).sum()}")
print(f"  Sleep day:      {sleep_day.duplicated(subset=['Id','SleepDay']).sum()}")
print(f"  Hourly steps:   {hourly_steps.duplicated(subset=['Id','ActivityHour']).sum()}")

# Check WHICH dates have daily activity duplicates
dup_mask = daily_activity.duplicated(subset=['Id','ActivityDate'], keep=False)
print(f"\nDaily activity duplicates are all on: {daily_activity.loc[dup_mask, 'ActivityDate'].unique()}")

All 24 daily-activity duplicates are on **April 12, 2016** — the boundary between the two folders. Folder 1 ends 4/11 and Folder 2 starts 4/12, but they overlap on 4/12.

Looking at one user's duplicate pair shows what's going on: Folder 1 has a row with 224 steps and 50 calories for 4/12, while Folder 2 has 13,162 steps and 1,985 calories for the same user, same day. The Folder 1 row is a **partial day** — the data extraction got cut off mid-day on 4/12.

So we want to **keep the full-day records** (Folder 2) and drop the partial-day records (Folder 1).

## 5. Clean: drop partial-day records, then true duplicates

Strategy:
- **Daily activity:** for any (Id, date) duplicate, keep the row with the higher TotalSteps. The partial-day rows always have lower step counts because they cover fewer hours.
- **Sleep:** the 3 duplicates are truly identical — just drop them.
- **Hourly steps:** the 175 duplicates are also from the 4/12 cutoff, but in this case both folders captured the same hours with the same values, so they're safe to drop.

In [ ]:
before = len(daily_activity)
daily_activity = (
    daily_activity
    .sort_values('TotalSteps', ascending=False)
    .drop_duplicates(subset=['Id','ActivityDate'], keep='first')
    .sort_values(['Id','ActivityDate'])
    .reset_index(drop=True)
)
print(f"Daily activity: dropped {before - len(daily_activity)} partial-day rows")

before = len(sleep_day)
sleep_day = sleep_day.drop_duplicates().reset_index(drop=True)
print(f"Sleep day:      dropped {before - len(sleep_day)} duplicate rows")

before = len(hourly_steps)
hourly_steps = hourly_steps.drop_duplicates().reset_index(drop=True)
print(f"Hourly steps:   dropped {before - len(hourly_steps)} duplicate rows")

## 6. Convert dates to proper datetime

All date columns came in as strings. Specifying the format explicitly keeps things fast and avoids parser warnings.

In [ ]:
daily_activity['ActivityDate'] = pd.to_datetime(
    daily_activity['ActivityDate'], format='%m/%d/%Y'
)
sleep_day['SleepDay'] = pd.to_datetime(
    sleep_day['SleepDay'], format='%m/%d/%Y %I:%M:%S %p'
).dt.date
hourly_steps['ActivityHour'] = pd.to_datetime(
    hourly_steps['ActivityHour'], format='%m/%d/%Y %I:%M:%S %p'
)

print("Date ranges:")
print(f"  Daily activity: {daily_activity['ActivityDate'].min().date()} to {daily_activity['ActivityDate'].max().date()}")
print(f"  Sleep day:      {sleep_day['SleepDay'].min()} to {sleep_day['SleepDay'].max()}")
print(f"  Hourly steps:   {hourly_steps['ActivityHour'].min()} to {hourly_steps['ActivityHour'].max()}")

## 7. Filter out non-wear days

Days with under 100 steps are almost certainly days the user didn't wear the device. Keeping them in averages would drag everything down. We keep `daily_activity` intact (for engagement metrics) and create `daily_activity_worn` for activity analysis.

In [ ]:
print("Non-wear day analysis:")
print(f"  Days with 0 steps:        {(daily_activity['TotalSteps'] == 0).sum()}")
print(f"  Days with < 100 steps:    {(daily_activity['TotalSteps'] < 100).sum()}")
print(f"  Total days in dataset:    {len(daily_activity)}")

daily_activity_worn = daily_activity[daily_activity['TotalSteps'] >= 100].copy()
print(f"  After non-wear filter:    {len(daily_activity_worn)}")

**11% of recorded days are non-wear** — itself a finding about engagement. People put their tracker in a drawer one day in nine.

## 8. Classify users into activity tiers

Using Tudor-Locke step thresholds from public-health research, not arbitrary cutoffs:

| Tier | Daily steps |
|---|---|
| Sedentary | < 5,000 |
| Lightly Active | 5,000 – 7,499 |
| Fairly Active | 7,500 – 9,999 |
| Very Active | 10,000+ |

In [ ]:
user_avg_steps = daily_activity_worn.groupby('Id')['TotalSteps'].mean().sort_values()

def classify_user(avg_steps):
    if avg_steps < 5000:
        return "Sedentary"
    elif avg_steps < 7500:
        return "Lightly Active"
    elif avg_steps < 10000:
        return "Fairly Active"
    else:
        return "Very Active"

user_segments = user_avg_steps.apply(classify_user)
print("User activity tiers:")
print(user_segments.value_counts())
print("\nAs percentages:")
print((user_segments.value_counts(normalize=True) * 100).round(1))

**Finding 1:** Users split 23/29/23/26% across the four tiers. No tier holds more than 30% of users — meaning there's no "typical" user to design a single marketing message around.

## 9. How users actually spend their day

The daily activity file gives us minutes spent in each intensity. What does the average day look like?

In [ ]:
intensity_cols = ['VeryActiveMinutes', 'FairlyActiveMinutes', 'LightlyActiveMinutes', 'SedentaryMinutes']
avg_minutes = daily_activity_worn[intensity_cols].mean()
total_minutes = avg_minutes.sum()

print("Average minutes per day on worn days:")
print(avg_minutes.round(1))
print(f"\nTotal tracked minutes/day: {total_minutes:.0f} ({total_minutes/60:.1f} hrs)")
print("\nPercentage breakdown:")
print((avg_minutes / total_minutes * 100).round(1))

**Finding 2:** People spend ~957 sedentary minutes (16 hours) per day vs. only 37 minutes in genuinely active states. That's a **26-to-1 ratio**. Even users in the Very Active tier sit most of the day.

This was the most surprising finding. The conventional fitness app pitch is "track your workouts." But the math says the bigger health lever is breaking up sitting time, not adding more workouts.

## 10. Sleep analysis

Convert minutes to hours for readability and add sleep efficiency (time asleep / time in bed).

In [ ]:
sleep_day['HoursAsleep']     = sleep_day['TotalMinutesAsleep'] / 60
sleep_day['HoursInBed']      = sleep_day['TotalTimeInBed'] / 60
sleep_day['SleepEfficiency'] = (sleep_day['TotalMinutesAsleep'] / sleep_day['TotalTimeInBed'] * 100).round(1)

print("Sleep summary:")
print(sleep_day[['HoursAsleep','HoursInBed','SleepEfficiency']].describe().round(2))

user_avg_sleep = sleep_day.groupby('Id')['HoursAsleep'].mean()
print(f"\nUsers averaging >= 7 hours: {(user_avg_sleep >= 7).sum()} of {len(user_avg_sleep)}")
print(f"Users averaging < 6 hours:  {(user_avg_sleep < 6).sum()} of {len(user_avg_sleep)}")

**Finding 4 (the headline twist):** Average sleep across all records is 6.99 hours — sounds like users are basically meeting the 7-hour recommendation. They're not. Only **46% of sleep-tracking users** actually average 7+ hours. **33%** are under 6.

But sleep efficiency is **91.6%** — high. Users aren't tossing and turning. They're just not allocating enough time to sleep in the first place. That changes what the right app feature should be: "go to bed earlier" instead of "improve your sleep quality."

## 11. Does activity correlate with sleep?

Merge daily activity with sleep on (user, date) and check correlations.

In [ ]:
daily_activity_worn['ActivityDate'] = pd.to_datetime(daily_activity_worn['ActivityDate']).dt.date

merged = daily_activity_worn.merge(
    sleep_day,
    left_on=['Id','ActivityDate'],
    right_on=['Id','SleepDay'],
    how='inner'
)

print(f"Merged dataset: {len(merged)} user-days, {merged['Id'].nunique()} users")

correlations = merged[['TotalSteps','VeryActiveMinutes','SedentaryMinutes','TotalMinutesAsleep']].corr()['TotalMinutesAsleep'].round(3)
print("\nCorrelation with TotalMinutesAsleep:")
print(correlations)

**Finding 5 (the strongest signal in the data):** Sedentary minutes correlates **−0.64** with sleep — a strong negative relationship. Steps barely correlate (−0.20). Very-active minutes basically don't correlate at all (−0.09).

It's specifically how much someone *sits* that lines up with bad sleep, not lack of fitness. Correlation isn't causation — but for marketing purposes, the implication holds: a feature that helps people sit less might also help them sleep more.

## 12. When are users most active?

This drives notification timing. If we know when activity peaks and dips, we know when nudges will actually land.

In [ ]:
hourly_steps['Hour'] = hourly_steps['ActivityHour'].dt.hour
avg_steps_by_hour = hourly_steps.groupby('Hour')['StepTotal'].mean().round(0)

print("Average steps by hour:")
print(avg_steps_by_hour)
print(f"\nPeak hour: {avg_steps_by_hour.idxmax()}:00 ({int(avg_steps_by_hour.max())} steps)")
print(f"Lowest:    {avg_steps_by_hour.idxmin()}:00 ({int(avg_steps_by_hour.min())} steps)")

**Finding 3:** Activity peaks at 7 PM (555 steps), with a clear lull from 1–4 PM and a sharp drop after 8 PM. The afternoon dip and evening wind-down both line up nicely with notification windows — movement nudges in the dead zone, bedtime prompts when activity is already declining.

## 13. Engagement / wear consistency

In [ ]:
days_per_user = daily_activity.groupby('Id').size().sort_values()
print("Days of data per user (out of 61 possible):")
print(days_per_user.describe().round(1))
print(f"\nUsers with fewer than 30 days: {(days_per_user < 30).sum()}")

**Finding 6:** The dataset covers 61 days, but the average user only has 39 days of data — 64% engagement. Two users had under 10 days, basically abandoning the device early. Combined with the 11% non-wear rate, there's a real engagement decay problem worth designing around.

## 14. Export the cleaned data

In [ ]:
daily_activity_worn.to_csv(CLEAN_DIR / "daily_activity_cleaned.csv", index=False)
sleep_day.to_csv(CLEAN_DIR / "sleep_day_cleaned.csv", index=False)
hourly_steps.to_csv(CLEAN_DIR / "hourly_steps_cleaned.csv", index=False)
merged.to_csv(CLEAN_DIR / "activity_sleep_merged.csv", index=False)
print(f"Cleaned data exported to: {CLEAN_DIR}")

## 15. Visualizations

Six charts, one per finding. Bellabeat-aligned palette: warm coral primary, sage secondary, deep slate dark.

In [ ]:
BELLA = {
    'primary':   '#E07A5F',
    'secondary': '#81B29A',
    'accent':    '#F2CC8F',
    'dark':      '#3D405B',
    'light':     '#F4F1DE'
}

sns.set_style('whitegrid')
plt.rcParams['figure.figsize']     = (10, 6)
plt.rcParams['font.size']          = 11
plt.rcParams['axes.titlesize']     = 14
plt.rcParams['axes.titleweight']   = 'bold'
plt.rcParams['axes.spines.top']    = False
plt.rcParams['axes.spines.right']  = False

### Chart 1 — User segments

In [ ]:
segment_order  = ['Sedentary', 'Lightly Active', 'Fairly Active', 'Very Active']
segment_counts = user_segments.value_counts().reindex(segment_order)

fig, ax = plt.subplots()
bars = ax.bar(segment_counts.index, segment_counts.values,
              color=[BELLA['primary'], BELLA['accent'], BELLA['secondary'], BELLA['dark']])

for bar, count in zip(bars, segment_counts.values):
    pct = count / segment_counts.sum() * 100
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
            f'{count}\n({pct:.0f}%)', ha='center', va='bottom', fontsize=10)

ax.set_title('Users are evenly split across activity levels — no single "typical" user', loc='left')
ax.set_ylabel('Number of users')
ax.set_ylim(0, max(segment_counts.values) + 2)
plt.tight_layout()
plt.savefig(VIZ_DIR / "01_user_segments.png", dpi=150, bbox_inches='tight')
plt.show()

### Chart 2 — Daily activity composition (the 79% sedentary chart)

In [ ]:
labels  = ['Sedentary', 'Lightly Active', 'Fairly Active', 'Very Active']
minutes = avg_minutes.values
colors  = [BELLA['primary'], BELLA['accent'], BELLA['secondary'], BELLA['dark']]

fig, ax = plt.subplots(figsize=(10, 5))
left = 0
for lbl, mins, c in zip(labels, minutes, colors):
    pct = mins / minutes.sum() * 100
    ax.barh(['Average Day'], [mins], left=[left], color=c, label=f'{lbl} ({pct:.0f}%)')
    left += mins

ax.set_title('79% of the average tracked day is sedentary', loc='left')
ax.set_xlabel('Minutes per day')
ax.set_xlim(0, 1300)
ax.legend(loc='lower right', frameon=False)
plt.tight_layout()
plt.savefig(VIZ_DIR / "02_daily_composition.png", dpi=150, bbox_inches='tight')
plt.show()

### Chart 3 — Hourly step pattern

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(avg_steps_by_hour.index, avg_steps_by_hour.values,
        color=BELLA['primary'], linewidth=2.5, marker='o', markersize=5)
ax.fill_between(avg_steps_by_hour.index, avg_steps_by_hour.values,
                color=BELLA['primary'], alpha=0.15)

peak_hour  = avg_steps_by_hour.idxmax()
peak_value = avg_steps_by_hour.max()
ax.annotate(f'Peak: 7 PM\n{int(peak_value)} steps',
            xy=(peak_hour, peak_value),
            xytext=(peak_hour - 5, peak_value + 80),
            fontsize=10, color=BELLA['dark'],
            arrowprops=dict(arrowstyle='->', color=BELLA['dark']))

ax.axvspan(13, 16, alpha=0.1, color=BELLA['secondary'])
ax.text(14.5, 100, 'Afternoon lull\n(notification window)',
        ha='center', fontsize=9, color=BELLA['dark'], style='italic')

ax.set_title('Activity peaks at 7 PM, with an afternoon lull from 1–4 PM', loc='left')
ax.set_xlabel('Hour of day')
ax.set_ylabel('Average steps')
ax.set_xticks(range(0, 24, 2))
ax.set_xticklabels([f'{h}:00' for h in range(0, 24, 2)])
plt.tight_layout()
plt.savefig(VIZ_DIR / "03_hourly_pattern.png", dpi=150, bbox_inches='tight')
plt.show()

### Chart 4 — Sleep distribution per user

In [ ]:
user_avg_sleep_sorted = sleep_day.groupby('Id')['HoursAsleep'].mean().sort_values()

fig, ax = plt.subplots(figsize=(10, 6))
bar_colors = [BELLA['primary'] if hrs < 6 else BELLA['accent'] if hrs < 7 else BELLA['secondary']
              for hrs in user_avg_sleep_sorted.values]
ax.barh(range(len(user_avg_sleep_sorted)), user_avg_sleep_sorted.values, color=bar_colors)

ax.axvline(7, color=BELLA['dark'], linestyle='--', linewidth=1.5, alpha=0.7)
ax.text(7.05, len(user_avg_sleep_sorted) - 1, '7-hour minimum',
        fontsize=9, color=BELLA['dark'], style='italic')

ax.set_title('1 in 3 sleep-tracking users averages less than 6 hours per night', loc='left')
ax.set_xlabel('Average hours of sleep per night')
ax.set_ylabel('Users (anonymized, sorted)')
ax.set_yticks([])
ax.set_xlim(0, 10)

legend_elements = [
    Patch(facecolor=BELLA['primary'],   label='< 6 hrs (sleep-deprived)'),
    Patch(facecolor=BELLA['accent'],    label='6–7 hrs (under recommendation)'),
    Patch(facecolor=BELLA['secondary'], label='≥ 7 hrs (meets recommendation)')
]
ax.legend(handles=legend_elements, loc='lower right', frameon=False)
plt.tight_layout()
plt.savefig(VIZ_DIR / "04_sleep_distribution.png", dpi=150, bbox_inches='tight')
plt.show()

### Chart 5 — Sedentary vs sleep (the strongest signal)

In [ ]:
sed_hours   = merged['SedentaryMinutes'] / 60
sleep_hours = merged['TotalMinutesAsleep'] / 60

fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(sed_hours, sleep_hours, color=BELLA['primary'],
           alpha=0.5, s=40, edgecolor='white', linewidth=0.5)

z = np.polyfit(sed_hours, sleep_hours, 1)
p = np.poly1d(z)
x_line = np.linspace(sed_hours.min(), sed_hours.max(), 100)
ax.plot(x_line, p(x_line), color=BELLA['dark'], linewidth=2, linestyle='--')

ax.text(0.05, 0.05, 'Correlation: −0.64\n(strong negative)',
        transform=ax.transAxes, fontsize=11,
        bbox=dict(boxstyle='round,pad=0.5', facecolor=BELLA['light'], edgecolor='none'))

ax.set_title('Users who sit more sleep less — the strongest relationship in the data', loc='left')
ax.set_xlabel('Sedentary hours per day')
ax.set_ylabel('Hours of sleep')
plt.tight_layout()
plt.savefig(VIZ_DIR / "05_sedentary_vs_sleep.png", dpi=150, bbox_inches='tight')
plt.show()

### Chart 6 — Wear consistency

In [ ]:
TOTAL_POSSIBLE_DAYS = 61

fig, ax = plt.subplots(figsize=(10, 6))
bar_colors = [BELLA['primary'] if d < 30 else BELLA['accent'] if d < 50 else BELLA['secondary']
              for d in days_per_user.values]
ax.barh(range(len(days_per_user)), days_per_user.values, color=bar_colors)

ax.axvline(TOTAL_POSSIBLE_DAYS, color=BELLA['dark'],
           linestyle='--', linewidth=1.5, alpha=0.7)
ax.text(TOTAL_POSSIBLE_DAYS + 0.5, len(days_per_user) - 1,
        f'{TOTAL_POSSIBLE_DAYS} days possible',
        fontsize=9, color=BELLA['dark'], style='italic')

ax.set_title('Users only had data for 39 of 61 days on average — engagement decays quickly', loc='left')
ax.set_xlabel('Days with recorded data')
ax.set_ylabel('Users (anonymized, sorted)')
ax.set_yticks([])
ax.set_xlim(0, 70)

legend_elements = [
    Patch(facecolor=BELLA['primary'],   label='< 30 days (low engagement)'),
    Patch(facecolor=BELLA['accent'],    label='30–49 days (moderate)'),
    Patch(facecolor=BELLA['secondary'], label='50+ days (high engagement)')
]
ax.legend(handles=legend_elements, loc='lower right', frameon=False)
plt.tight_layout()
plt.savefig(VIZ_DIR / "06_wear_consistency.png", dpi=150, bbox_inches='tight')
plt.show()

## 16. Summary

Everything ties back to five marketing suggestions for the Bellabeat app:

1. **Build different messaging for different user types** — users split evenly across four tiers, so one voice can't reach more than 25% of the audience.
2. **Lean into reducing sitting time instead of pushing workouts** — 79% of the day is sedentary, and sitting is the biggest correlate of bad sleep.
3. **Time notifications around the 1–4 PM lull and 9–10 PM wind-down** — when users are most reachable.
4. **Make sleep duration coaching a real feature** — a third of users get under 6 hours, but sleep efficiency is high. The fix is bedtime, not biology.
5. **Build a re-engagement flow for lapsed users** — average user only has 39 of 61 possible days of data.

See the accompanying report and deck for the full write-up and recommendations.